# 🔹 Escoamento em Tubulações e Bombeamento
## Volume I – Capítulo 4 | Nível: Graduação/Pós-Graduação

**Objetivos de aprendizagem:**
- Aplicar a equação de Bernoulli generalizada
- Calcular perda de carga distribuída (Darcy-Weisbach)
- Resolver Colebrook-White por Newton-Raphson
- Dimensionar potência de bombas e custos energéticos

**Referência:** Lugon Junior (2026), Vol I, Cap. 4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import newton

# Constantes
G = 9.81
NU_AGUA = 1e-6
RHO_AGUA = 998.0
GAMMA = RHO_AGUA * G

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 🔸 Fator de Atrito: Swamee-Jain vs Colebrook-White

In [ ]:
def colebrook_white(f, Re, eps_D):
    """Função implícita de Colebrook-White."""
    return 1/np.sqrt(f) + 2*np.log10(eps_D/3.7 + 2.51/(Re*np.sqrt(f)))

def swamee_jain(Re, eps_D):
    """Correlação explícita de Swamee-Jain."""
    return 0.25 / (np.log10(eps_D/3.7 + 5.74/Re**0.9))**2

def calcular_f(Re, eps_D, metodo='swamee'):
    """Calcula fator de atrito f."""
    if Re < 2000:
        return 64/Re  # Laminar
    elif metodo == 'colebrook':
        return newton(colebrook_white, 0.02, args=(Re, eps_D), tol=1e-8)
    else:
        return swamee_jain(Re, eps_D)

# Exercício Vol I, Cap 4, Ex 1
Q = 0.003        # m³/s (3 L/s)
D = 0.075        # m
L = 80.0         # m
eps = 0.00015    # m (ferro galvanizado)
nu = 1e-6        # m²/s

# Cálculos básicos
A = np.pi * D**2 / 4
V = Q / A
Re = V * D / nu
eps_D = eps / D

f_swa = calcular_f(Re, eps_D, 'swamee')
f_col = calcular_f(Re, eps_D, 'colebrook')

print(f"Velocidade: V = {V:.3f} m/s")
print(f"Reynolds: Re = {Re:.2e} → TURBULENTO")
print(f"Rugosidade relativa: ε/D = {eps_D:.4f}")
print(f"\nFator de atrito:")
print(f"  Swamee-Jain:  f = {f_swa:.4f}")
print(f"  Colebrook:    f = {f_col:.4f}")
print(f"  Erro relativo: {(f_col-f_swa)/f_col*100:.2f}%")

## 🔸 Perda de Carga Total

In [ ]:
def perda_carga(Q, D, L, eps, nu=NU_AGUA, K_local=0, metodo='swamee'):
    """
    Calcula perda de carga total (distribuída + localizada).
    """
    A = np.pi * D**2 / 4
    V = Q / A
    Re = V * D / nu
    eps_D = eps / D
    
    f = calcular_f(Re, eps_D, metodo)
    
    # Distribuída (Darcy-Weisbach)
    hf_dist = f * (L/D) * (V**2 / (2*G))
    
    # Localizada
    hf_loc = K_local * (V**2 / (2*G))
    
    return hf_dist, hf_loc, hf_dist + hf_loc, f, V, Re

# Caso do exercício
K_total = 2*0.15 + 6*0.9 + 2.5  # válvulas + curvas + retenção
hf_d, hf_l, hf_tot, f, V, Re = perda_carga(Q, D, L, eps, K_local=K_total)

print(f"\nPerda de carga:")
print(f"  Distribuída:  h_f = {hf_d:.3f} m")
print(f"  Localizada:   h_s = {hf_l:.3f} m")
print(f"  TOTAL:        h_f,total = {hf_tot:.3f} m")

## 🔸 Potência e Custo de Bombeamento

In [ ]:
def custo_bombeamento(Q, H_B, eta, horas_dia, tarifa, dias=30):
    """
    Calcula potência e custo energético de bombeamento.
    """
    P_hid = GAMMA * Q * H_B  # W
    P_el = P_hid / eta        # W
    
    energia_kWh = (P_el / 1000) * horas_dia * dias
    custo = energia_kWh * tarifa
    
    return P_el/1000, energia_kWh, custo

# Exercício Vol I, Cap 4, Ex 2
H_B = 30.0          # m
eta = 0.70          # rendimento
horas = 6           # h/dia
tarifa = 0.90       # R$/kWh

P_kw, E_kWh, custo = custo_bombeamento(Q, H_B, eta, horas, tarifa)

print(f"\nBombeamento (Q={Q*1000:.1f} L/s, H_B={H_B} m):")
print(f"  Potência elétrica:  P_el = {P_kw:.3f} kW")
print(f"  Energia mensal:     E = {E_kWh:.1f} kWh")
print(f"  Custo mensal:       R$ {custo:.2f}")

# Varredura paramétrica: custo vs vazão
Q_vals = np.linspace(0.001, 0.010, 50)
custos = [custo_bombeamento(q, H_B, eta, horas, tarifa)[2] for q in Q_vals]

plt.figure(figsize=(8, 5))
plt.plot(Q_vals*1000, custos, 'b-', linewidth=2)
plt.xlabel('Vazão Q [L/s]')
plt.ylabel('Custo mensal [R$]')
plt.title('Custo energético vs Vazão (H_B = 30 m, η = 70%)')
plt.grid(True, alpha=0.3)
plt.axvline(x=Q*1000, color='r', linestyle='--', label='Caso base')
plt.legend()
plt.tight_layout()
plt.show()

## ✅ Exercícios Propostos

1. **[Graduação]** Calcule a perda localizada para um sistema com 2 válvulas de gaveta, 6 curvas 90°, 1 válvula de retenção, V = 2 m/s.
2. **[Pós-Graduação]** Implemente Newton-Raphson para Colebrook-White e compare convergência com Swamee-Jain para Re = 10⁴, 10⁵, 10⁶.

> 💡 Dica: Use `scipy.optimize.newton` com derivada analítica para acelerar a convergência.